# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 03 · Modelos clásicos

Entrena o audita el contrato de etiquetas v2.1 sin consultar test para seleccionar modelos, épocas o umbrales.

La suite representa texto mediante TF–IDF [1] y compara regresión logística [2], SVM lineal [3], Complement Naive Bayes [4] y descenso de gradiente estocástico [5]. La implementación usa scikit-learn [6] bajo una transformación uno-contra-resto para el problema multietiqueta [7]; la elección de n-gramas, pesos de clase e hiperparámetros es local.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [6]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import notebook_progress, run_with_progress, show_callout, show_command, show_result, show_summary, show_table
OPERATIONAL_PROMPT=ROOT/'config/prompt_operacional_ollama_v3_2.md'
if not OPERATIONAL_PROMPT.is_file():
    raise FileNotFoundError(f'Falta el prompt operacional vigente: {OPERATIONAL_PROMPT}')
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local', 'prompt_operacional': OPERATIONAL_PROMPT}, tone='success')


raíz,C:/usr/ths_mia_fiis/pln/trabajo
backend,local
prompt_operacional,C:/usr/ths_mia_fiis/pln/trabajo/config/prompt_operacional_ollama_v3_2.md


## Procedimiento reproducible por corridas

Este cuaderno se ejecuta **localmente** desde el repositorio completo. Todas las corridas deben usar el mismo dataset verificado: ejecute desde la primera celda y active una sola fase por vez. Cada fase escribe en un subdirectorio distinto y una firma idéntica produce `status=noop`.

### Ruta A · recuperar el entrenamiento local anterior

La ejecución guardada del cuaderno demuestra que el dataset usado tuvo SHA-256 `013d60ba1b173d7752f453d5d05629a3439b09c71f0c343da1b5e498662c1f86` y que la raíz anterior fue `D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4`. Los pesos no están en GitHub porque `modelos/*` está excluido; la salida HTML del cuaderno no permite reconstruirlos.

1. Ejecute la última celda con sus tres compuertas en `False` y revise `carpeta_actual_existe` y `carpeta_historica_existe`. Si la carpeta actual ya existe, no la copie: salte directamente a la auditoría.
2. Si la actual falta, compruebe `D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/modelos/v2/clasicos`. Si se movió, edite solo `LEGACY_CLASSICAL_ROOT`; luego active únicamente `RUN_RECOVER_LEGACY_03_01=True`. La celda copia sin reemplazar una salida actual y exige candidatos clásicos completos, con validation común, el mismo SHA-256 y test sellado.
3. Vuelva a dejar la recuperación en `False`; active solo `RUN_AUDIT_03_01=True`. No continúe si `listo_para_publicar` no es `true`.

### Ruta B · reconstruir si la carpeta anterior ya no existe

1. **Suite principal.** Active solo `RUN_TRAINING=True`. Esta es la corrida comparable que crea los candidatos base e informados por política. Al terminar vuelva a `False`.
2. **Reparación SVM.** Ejecútela solo si el candidato SVM informa convergencia no verificada: mantenga la suite en `False` y active `RUN_SVM_CONVERGENCE_REPAIR=True`. No sustituye silenciosamente los otros estimadores.
3. **Robustez por canal opcional.** Active solo `RUN_CHANNEL_ROBUSTNESS=True`. Es un diagnóstico con otra partición y no entra en la comparación común de `03_07`; no es necesario para recuperar la publicación.
4. Deje las tres fases de entrenamiento en `False` y ejecute únicamente `RUN_AUDIT_03_01=True` en la celda de recuperación/publicación.

### Mejora acotada · regularización de los dos modelos lineales

Esta fase no repite la suite completa. Mantiene la variante TF-IDF `base`, las mismas filas, semilla, muestreo 4:1 y test sellado; ajusta únicamente la intensidad inversa de regularización `C`.

1. Deje `RUN_TRAINING=False`, `RUN_SVM_CONVERGENCE_REPAIR=False` y `RUN_CHANNEL_ROBUSTNESS=False`.
2. En **Barrido acotado de regularización**, active solo `RUN_REGULARIZATION_SCREEN=True`. Se entrenan seis candidatos: regresión logística y SVM lineal para `C = 0.5, 1.0, 2.0`. El TF-IDF se extrae una vez y se comparte entre los seis.
3. Espere la tabla completa y confirme `convergencia=True` en los tres SVM. Una repetición con la misma firma devuelve `status=noop`; no use `force`.
4. Vuelva a `RUN_REGULARIZATION_SCREEN=False`, active únicamente `RUN_AUDIT_03_01=True` y prepare una publicación nueva. `03_07` comparará estos candidatos con los históricos usando la misma validation.

No aumente `max_iter` si los SVM convergen: `20000` es un techo, no una cantidad obligatoria de iteraciones. Tampoco cambie `C`, TF-IDF y muestreo en una misma campaña.

### Publicación local para `03_07` sin Drive para escritorio

1. Después de una auditoría correcta, deje `RUN_AUDIT_03_01=False` y active solo `RUN_PREPARE_03_01_DRIVE_PUBLICATION=True`. La celda empaqueta los artefactos en dos ranuras, relee el TAR completo, verifica tamaño y SHA-256, lo restaura en un directorio temporal y repite la auditoría.
2. Suba mediante <https://drive.google.com> la **carpeta completa** `resultados/drive_staging/ModeracionPeru_Colab/runs/03_01/03_01_working_v2_1` a `Mi unidad/ModeracionPeru_Colab/runs/03_01/`. El resultado exacto debe ser `Mi unidad/ModeracionPeru_Colab/runs/03_01/03_01_working_v2_1/run_manifest.json` junto con `publications/`; no seleccione archivos sueltos.
3. Abra una copia nueva de `03_07` en **Colab web**, runtime CPU, y ejecute el preflight desde la primera celda. `03_01` debe aparecer como `restored_and_sha256_verified` y la familia `classical:` no debe figurar como ausente.
4. Al terminar deje `RUN_REGULARIZATION_SCREEN`, `RUN_RECOVER_LEGACY_03_01`, `RUN_AUDIT_03_01` y `RUN_PREPARE_03_01_DRIVE_PUBLICATION` en `False`.

No mueva candidatos entre snapshots, no reconstruya pesos desde las métricas impresas y no copie métricas del diagnóstico por canal al ranking principal.

## Restauración reproducible del dataset

In [7]:
from moderacion_peru.colab import prepare_local_bundle_input

if globals().get('COLAB_CONTEXT') is None:
    dataset_checkpoint = prepare_local_bundle_input('dataset_5_salidas', project_root=ROOT)
else:
    dataset_path = COLAB_CONTEXT.input('dataset_5_salidas')
    dataset_checkpoint = {
        'status': 'verified_in_colab',
        'input_key': 'dataset_5_salidas',
        'path': dataset_path,
        'bytes': dataset_path.stat().st_size,
    }
show_result('Dataset descomprimido y verificado', dataset_checkpoint, tone='success')


status,verified_existing
input_key,dataset_5_salidas
path,C:/usr/ths_mia_fiis/pln/trabajo/datos/model_ready/v2/dataset_5_salidas.jsonl
sha256,013d60ba1b173d7752f453d5d05629a3439b09c71f0c343da1b5e498662c1f86
bytes,225478048
archive,C:/usr/ths_mia_fiis/pln/trabajo/resultados/colab_bundle/dataset_5_salidas.jsonl.gz


## Configuración y ejecución

In [8]:
from moderacion_peru.experiments import train_classical_experiments
DATA=ROOT/'datos/model_ready/v2/dataset_5_salidas.jsonl'
OUTPUT=ROOT/'modelos/v2/clasicos'
SAFE_TO_DAMAGE_RATIO=4.0  # política fija en train y validation
PARALLEL_WORKERS=4  # 4/16 hilos: comparte la matriz dispersa sin saturar RAM
LINEAR_SVM_MAX_ITER=20000  # 1.000 produjo ConvergenceWarning en 22 salidas × 3 pliegues
RUN_TRAINING=False
RUN_SVM_CONVERGENCE_REPAIR=False
RUN_CHANNEL_ROBUSTNESS=False
if RUN_TRAINING:
    classical_result=run_with_progress('Suite clásica',train_classical_experiments,DATA,OUTPUT,variants=('base','policy_informed'),safe_to_damage_ratio=SAFE_TO_DAMAGE_RATIO,parallel_workers=PARALLEL_WORKERS,linear_svm_max_iter=LINEAR_SVM_MAX_ITER,progress_unit='etapa')
    show_result('Clásicos base e informados por política',classical_result,tone='success')
if RUN_SVM_CONVERGENCE_REPAIR:
    svm_result=run_with_progress('Reparación SVM',train_classical_experiments,DATA,OUTPUT/'svm_convergence_repair',model_names=('linear_svm',),variants=('base','policy_informed'),safe_to_damage_ratio=SAFE_TO_DAMAGE_RATIO,parallel_workers=PARALLEL_WORKERS,linear_svm_max_iter=LINEAR_SVM_MAX_ITER,progress_unit='etapa')
    show_result('SVM recalibrado con convergencia verificable',svm_result,tone='warning' if any(candidate['fit_quality']['converged'] is not True for candidate in svm_result['candidates']) else 'success')
if RUN_CHANNEL_ROBUSTNESS:
    robustness_result=run_with_progress('Robustez por canal',train_classical_experiments,DATA,OUTPUT/'channel_heldout',model_names=('logistic_regression',),variants=('base','policy_informed'),safe_to_damage_ratio=SAFE_TO_DAMAGE_RATIO,split_scheme='channel',parallel_workers=PARALLEL_WORKERS,progress_unit='etapa')
    show_result('Robustez con canales retenidos',robustness_result,tone='success')
if not (RUN_TRAINING or RUN_SVM_CONVERGENCE_REPAIR or RUN_CHANNEL_ROBUSTNESS):
    show_summary('Entrenamiento desactivado',{'salidas':'22 enmascaradas','SEGURO_train_validation':'4:1','TF-IDF':'una extracción por variante, reutilizada por cinco modelos','SVM':f'max_iter={LINEAR_SVM_MAX_ITER}; convergencia registrada y exigida en 03_07','paralelismo':f'{PARALLEL_WORKERS} hilos compartiendo matriz dispersa','progreso':'barra por preparación, TF-IDF, candidato y validation','test':'natural completo, sellado'},tone='neutral')

salidas,22 enmascaradas
SEGURO_train_validation,4:1
TF-IDF,"una extracción por variante, reutilizada por cinco modelos"
SVM,max_iter=20000; convergencia registrada y exigida en 03_07
paralelismo,4 hilos compartiendo matriz dispersa
progreso,"barra por preparación, TF-IDF, candidato y validation"
test,"natural completo, sellado"


## Barrido acotado de regularización

Compara regresión logística y SVM lineal con `C = 0.5, 1.0, 2.0`. Los seis candidatos reutilizan una sola extracción TF-IDF base, conservan las mismas filas y mantienen test sellado.

In [9]:
import importlib
import inspect
import moderacion_peru.experiments as classical_experiments_module

# Un notebook puede actualizarse mientras el kernel conserva la función anterior
# en memoria. Recarga únicamente si la API todavía no expone el barrido de C.
REGULARIZATION_API_RELOADED=False
if 'regularization_c_values' not in inspect.signature(train_classical_experiments).parameters:
    classical_experiments_module=importlib.reload(classical_experiments_module)
    train_classical_experiments=classical_experiments_module.train_classical_experiments
    REGULARIZATION_API_RELOADED=True
if 'regularization_c_values' not in inspect.signature(train_classical_experiments).parameters:
    raise RuntimeError(
        'El kernel sigue usando una API anterior. Reinicie el kernel y ejecute desde la primera celda.'
    )

REGULARIZATION_C_VALUES=(0.5,1.0,2.0)
REGULARIZATION_OUTPUT=OUTPUT/'regularization_screen'
RUN_REGULARIZATION_SCREEN=False

other_classical_phases_active=any((
    RUN_TRAINING,
    RUN_SVM_CONVERGENCE_REPAIR,
    RUN_CHANNEL_ROBUSTNESS,
))
if RUN_REGULARIZATION_SCREEN and other_classical_phases_active:
    raise ValueError(
        'Desactive RUN_TRAINING, RUN_SVM_CONVERGENCE_REPAIR y '
        'RUN_CHANNEL_ROBUSTNESS antes del barrido de C'
    )

if RUN_REGULARIZATION_SCREEN:
    regularization_result=run_with_progress(
        'Regularización acotada de regresión logística y SVM',
        train_classical_experiments,
        DATA,
        REGULARIZATION_OUTPUT,
        model_names=('logistic_regression','linear_svm'),
        variants=('base',),
        regularization_c_values=REGULARIZATION_C_VALUES,
        safe_to_damage_ratio=SAFE_TO_DAMAGE_RATIO,
        parallel_workers=PARALLEL_WORKERS,
        linear_svm_max_iter=LINEAR_SVM_MAX_ITER,
        progress_unit='etapa',
    )
    regularization_candidates=regularization_result['candidates']
    regularization_rows=[{
        'candidate_id':candidate['candidate_id'],
        'modelo':candidate['model_family'],
        'C':candidate['hyperparameters']['regularization_C'],
        'convergencia':candidate['fit_quality']['converged'],
        'macro_AUPRC_daño_validation':candidate['validation_metrics']['average_precision_macro_damage'],
        'falsos_seguros_sobre_daño':candidate['validation_metrics']['false_safe_rate_on_damage'],
        'carga_revision':candidate['validation_metrics']['review_load_rate'],
        'test':candidate['test_status'],
    } for candidate in regularization_candidates]
    show_result(
        'Barrido acotado de C',
        regularization_result,
        tone=(
            'success'
            if all(candidate['fit_quality']['converged'] is True for candidate in regularization_candidates)
            else 'warning'
        ),
    )
    show_table('Comparación de regularización en validation',regularization_rows,max_rows=6)
else:
    show_summary('Barrido de regularización desactivado',{
        'modelos':('logistic_regression','linear_svm'),
        'C':REGULARIZATION_C_VALUES,
        'candidatos':6,
        'variante':'base; mismo TF-IDF compartido por los seis candidatos',
        'selección':'solo validation; 03_07 conserva test sellado',
        'salida':REGULARIZATION_OUTPUT,
        'API_recargada_por_kernel_antiguo':REGULARIZATION_API_RELOADED,
    },tone='neutral')


modelos,"Ver detalle[ ""logistic_regression"", ""linear_svm"" ]"
C,"Ver detalle[ 0.5, 1.0, 2.0 ]"
candidatos,6
variante,base; mismo TF-IDF compartido por los seis candidatos
selección,solo validation; 03_07 conserva test sellado
salida,C:/usr/ths_mia_fiis/pln/trabajo/modelos/v2/clasicos/regularization_screen
API_recargada_por_kernel_antiguo,No


## Recuperación, auditoría y publicación verificable de 03_01

Use esta celda después del entrenamiento o para recuperar la carpeta histórica. Prepara una publicación local que se carga como carpeta completa mediante Google Drive web; no requiere Google Drive para escritorio.

In [ ]:
from pathlib import Path
import shutil
import tempfile

from moderacion_peru.colab import ColabContext, publish_colab_outputs, restore_colab_run_outputs
from moderacion_peru.device import resolve_device
from moderacion_peru.ensemble_evaluation import audit_validation_candidate_eligibility

# La salida histórica visible en la ejecución guardada del cuaderno. Si se movió,
# edite únicamente esta ruta; no reconstruya archivos a partir de la salida HTML.
LEGACY_CLASSICAL_ROOT=Path('D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/modelos/v2/clasicos')
CURRENT_CLASSICAL_ROOT=ROOT/'modelos/v2/clasicos'
LOCAL_DRIVE_STAGING_ROOT=ROOT/'resultados/drive_staging/ModeracionPeru_Colab'
LOCAL_PUBLICATION_RUNTIME=ROOT/'resultados/local_publication_runtime'
DRIVE_NOTEBOOK_ID='03_01'
DRIVE_RUN_ID='03_01_working_v2_1'
DRIVE_RUN_DIR=LOCAL_DRIVE_STAGING_ROOT/'runs'/DRIVE_NOTEBOOK_ID/DRIVE_RUN_ID
DRIVE_WEB_DESTINATION='Mi unidad/ModeracionPeru_Colab/runs/03_01/03_01_working_v2_1'

# Active una sola compuerta por ejecución y vuelva a dejarla en False al terminar.
RUN_RECOVER_LEGACY_03_01=False
RUN_AUDIT_03_01=False
RUN_PREPARE_03_01_DRIVE_PUBLICATI"""  """ON=True

active_recovery_actions=sum((
    RUN_RECOVER_LEGACY_03_01,
    RUN_AUDIT_03_01,
    RUN_PREPARE_03_01_DRIVE_PUBLICATION,
))
if active_recovery_actions>1:
    raise ValueError('Active una sola compuerta de recuperación/publicación de 03_01 por vez')

def audit_classical_candidates(root):
    audit=audit_validation_candidate_eligibility(DATA,(root,))
    eligible_classical=[
        row for row in audit['eligible']
        if str(row.get('model_family','')).casefold().startswith('classical:')
    ]
    return audit,eligible_classical

show_summary('Preflight de recuperación y publicación de 03_01',{
    'carpeta_actual':CURRENT_CLASSICAL_ROOT,
    'carpeta_actual_existe':CURRENT_CLASSICAL_ROOT.is_dir(),
    'carpeta_historica':LEGACY_CLASSICAL_ROOT,
    'carpeta_historica_existe':LEGACY_CLASSICAL_ROOT.is_dir(),
    'staging_local_para_drive_web':DRIVE_RUN_DIR,
    'destino_en_drive_web':DRIVE_WEB_DESTINATION,
    'acciones_activas':active_recovery_actions,
},tone='neutral')

if RUN_RECOVER_LEGACY_03_01:
    if not LEGACY_CLASSICAL_ROOT.is_dir():
        raise FileNotFoundError(
            f'No existe {LEGACY_CLASSICAL_ROOT}. Edite LEGACY_CLASSICAL_ROOT si la carpeta se movió; '
            'si fue eliminada, ejecute nuevamente la suite principal.'
        )
    if CURRENT_CLASSICAL_ROOT.exists():
        raise FileExistsError(
            f'{CURRENT_CLASSICAL_ROOT} ya existe. No se reemplazará ni mezclará automáticamente; '
            'audítela o elija deliberadamente otra carpeta de destino.'
        )
    CURRENT_CLASSICAL_ROOT.parent.mkdir(parents=True,exist_ok=True)
    shutil.copytree(LEGACY_CLASSICAL_ROOT,CURRENT_CLASSICAL_ROOT)
    recovery_audit,recovery_eligible=audit_classical_candidates(CURRENT_CLASSICAL_ROOT)
    show_result('Carpeta histórica de 03_01 recuperada y auditada',{
        'origen':LEGACY_CLASSICAL_ROOT,
        'destino':CURRENT_CLASSICAL_ROOT,
        'descubiertos':recovery_audit['discovered_count'],
        'elegibles_clasicos':[row.get('candidate_id') for row in recovery_eligible],
        'rechazados':recovery_audit['rejected'],
    },tone='success' if recovery_eligible else 'warning')
    if not recovery_eligible:
        raise ValueError(
            'La copia histórica no contiene candidatos clásicos elegibles para el dataset activo. '
            'Conserve la auditoría y reconstruya 03_01 mediante las fases de entrenamiento indicadas.'
        )

if RUN_AUDIT_03_01:
    current_audit,current_eligible=audit_classical_candidates(CURRENT_CLASSICAL_ROOT)
    show_result('Auditoría local de 03_01',{
        'dataset_sha256':current_audit['dataset_sha256'],
        'descubiertos':current_audit['discovered_count'],
        'elegibles_clasicos':[{
            'candidate_id':row.get('candidate_id'),
            'model_family':row.get('model_family'),
        } for row in current_eligible],
        'rechazados':current_audit['rejected'],
        'listo_para_publicar':bool(current_eligible),
    },tone='success' if current_eligible else 'warning')
    if not current_eligible:
        raise ValueError(
            '03_01 aún no tiene un candidato clásico completo con validation común y test sellado.'
        )

if RUN_PREPARE_03_01_DRIVE_PUBLICATION:
    publication_audit,publication_eligible=audit_classical_candidates(CURRENT_CLASSICAL_ROOT)
    if not publication_eligible:
        raise ValueError(
            'No se publicará: primero complete RUN_AUDIT_03_01 con al menos un candidato clásico elegible.'
        )
    LOCAL_DRIVE_STAGING_ROOT.mkdir(parents=True,exist_ok=True)
    LOCAL_PUBLICATION_RUNTIME.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory(
        prefix='03_01-publication-payload-',dir=LOCAL_PUBLICATION_RUNTIME
    ) as scratch_name:
        scratch_output_dir=Path(scratch_name)
        shutil.copytree(CURRENT_CLASSICAL_ROOT,scratch_output_dir,dirs_exist_ok=True)
        local_context=ColabContext(
            notebook_id=DRIVE_NOTEBOOK_ID,
            run_id=DRIVE_RUN_ID,
            drive_root=LOCAL_DRIVE_STAGING_ROOT,
            runtime_root=LOCAL_PUBLICATION_RUNTIME,
            project_root=ROOT,
            input_paths={'dataset_5_salidas':DATA},
            scratch_output_dir=scratch_output_dir,
            drive_run_dir=DRIVE_RUN_DIR,
            hardware=resolve_device('cpu').model_dump(mode='json'),
            resumed=False,
        )
        publication_manifest=publish_colab_outputs(local_context)
    with tempfile.TemporaryDirectory(
        prefix='03_01-publication-verify-',dir=LOCAL_PUBLICATION_RUNTIME
    ) as verification_name:
        verification_root=Path(verification_name)
        restoration=restore_colab_run_outputs(
            LOCAL_DRIVE_STAGING_ROOT,
            notebook_id=DRIVE_NOTEBOOK_ID,
            run_id=DRIVE_RUN_ID,
            destination=verification_root,
        )
        verification_audit,verification_eligible=audit_classical_candidates(verification_root)
    if not verification_eligible:
        raise ValueError('La publicación se restauró, pero perdió la elegibilidad de los candidatos clásicos')
    show_result('Publicación local de 03_01 restaurada y verificada',{
        'manifest':publication_manifest,
        'restauracion_de_prueba':restoration,
        'dataset_sha256':verification_audit['dataset_sha256'],
        'candidatos_elegibles':[row.get('candidate_id') for row in verification_eligible],
        'carpeta_que_debe_subirse':DRIVE_RUN_DIR,
        'destino_exacto_en_drive_web':DRIVE_WEB_DESTINATION,
    },tone='success')
    show_callout(
        'Siguiente paso en Google Drive web',
        'Abra drive.google.com, cree Mi unidad/ModeracionPeru_Colab/runs/03_01 si falta y '
        'suba allí la carpeta completa 03_01_working_v2_1. No seleccione archivos sueltos y no '
        'necesita Google Drive para escritorio. Después abra una copia nueva de 03_07 en Colab web.',
        tone='success',
    )

if active_recovery_actions==0:
    show_callout(
        'Recuperación inactiva',
        'Elija una sola compuerta según el procedimiento: recuperar, auditar o preparar la publicación. '
        'Estas compuertas no entrenan modelos.',
        tone='neutral',
    )


carpeta_actual,C:/usr/ths_mia_fiis/pln/trabajo/modelos/v2/clasicos
carpeta_actual_existe,Sí
carpeta_historica,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/modelos/v2/clasicos
carpeta_historica_existe,No
staging_local_para_drive_web,C:/usr/ths_mia_fiis/pln/trabajo/resultados/drive_staging/ModeracionPeru_Colab/runs/03_01/03_01_working_v2_1
destino_en_drive_web,Mi unidad/ModeracionPeru_Colab/runs/03_01/03_01_working_v2_1
acciones_activas,1


manifest,"Ver detalle{ ""schema_version"": ""1.2.0"", ""published_at"": ""2026-08-14T21:46:59.396022+00:00"", ""notebook_id"": ""03_01"", ""run_id"": ""03_01_working_v2_1"", ""taxonomy_contract"": ""moderacion_peru_5_salidas_v2"", ""hardware"": { ""backend"": ""cpu"", ""requested"": ""cpu"", ""device_name"": ""Intel64 Family 6 Model 170 Stepping 4, GenuineIntel"", ""torch_version"": ""2.9.0+xpu"", ""runtime_version"": null, ""total_memory_bytes"": null, ""dtype"": ""float32"", ""fallback_reason"": null }, ""publication_slot"": ""a"", ""included_files"": 135, ""excluded_trainer_checkpoint_files"": 0, ""trainer_checkpoint_storage"": ""trainer_checkpoints"", ""archive"": { ""name"": ""publications/run_outputs-a.tar"", ""sha256"": ""7da0a79832228dc8788e27932f633e4dd779d9f58484a70c1afb86ff709b14bb"", ""bytes"": 1605488640, ""format"": ""tar_uncompressed"", ""verification"": ""full_readback_after_close"" } }"
restauracion_de_prueba,"Ver detalle{ ""status"": ""restored_and_sha256_verified"", ""source"": ""C:\\usr\\ths_mia_fiis\\pln\\trabajo\\resultados\\drive_staging\\ModeracionPeru_Colab\\runs\\03_01\\03_01_working_v2_1"", ""destination"": ""C:\\usr\\ths_mia_fiis\\pln\\trabajo\\resultados\\local_publication_runtime\\03_01-publication-verify-bqxtzsa8"" }"
dataset_sha256,013d60ba1b173d7752f453d5d05629a3439b09c71f0c343da1b5e498662c1f86
candidatos_elegibles,"Ver detalle[ ""classical-linear_svm_c0p5-54f7971c6000"", ""classical-linear_svm_c1-54f7971c6000"", ""classical-linear_svm_c2-54f7971c6000"", ""classical-logistic_regression_c0p5-54f7971c6000"", ""classical-logistic_regression_c1-54f7971c6000"", ""classical-logistic_regression_c2-54f7971c6000"", ""classical-base_complement_nb-347f3734fb9e"", ""classical-base_dummy-347f3734fb9e"", ""classical-base_linear_svm-347f3734fb9e"", ""classical-base_logistic_regression-347f3734fb9e"", ""classical-base_sgd_incremental-347f3734fb9e"", ""classical-policy_informed_complement_nb-347f3734fb9e"", ""classical-policy_informed_dummy-347f3734fb9e"", ""classical-policy_informed_linear_svm-347f3734fb9e"", ""classical-policy_informed_logistic_regression-347f3734fb9e"", ""classical-policy_informed_sgd_incremental-347f3734fb9e"" ]"
carpeta_que_debe_subirse,C:/usr/ths_mia_fiis/pln/trabajo/resultados/drive_staging/ModeracionPeru_Colab/runs/03_01/03_01_working_v2_1
destino_exacto_en_drive_web,Mi unidad/ModeracionPeru_Colab/runs/03_01/03_01_working_v2_1


## Referencias

[1] G. Salton and C. Buckley, "Term-Weighting Approaches in Automatic Text Retrieval," Inf. Process. Manage., vol. 24, no. 5, pp. 513–523, 1988, doi: 10.1016/0306-4573(88)90021-0.

[2] D. R. Cox, "The Regression Analysis of Binary Sequences," J. Roy. Stat. Soc. B, vol. 20, no. 2, pp. 215–232, 1958, doi: 10.1111/j.2517-6161.1958.tb00292.x.

[3] C. Cortes and V. Vapnik, "Support-Vector Networks," Mach. Learn., vol. 20, pp. 273–297, 1995, doi: 10.1007/BF00994018.

[4] J. D. M. Rennie, L. Shih, J. Teevan, et al., "Tackling the Poor Assumptions of Naive Bayes Text Classifiers," in Proc. ICML, 2003, pp. 616–623. [Online]. Available: https://people.csail.mit.edu/jrennie/papers/icml03-nb.pdf

[5] L. Bottou, "Large-Scale Machine Learning with Stochastic Gradient Descent," in Proc. COMPSTAT, 2010, pp. 177–186, doi: 10.1007/978-3-7908-2604-3_16.

[6] F. Pedregosa, G. Varoquaux, A. Gramfort, et al., "Scikit-Learn: Machine Learning in Python," J. Mach. Learn. Res., vol. 12, pp. 2825–2830, 2011. [Online]. Available: https://www.jmlr.org/papers/v12/pedregosa11a.html

[7] G. Tsoumakas and I. Katakis, "Multi-Label Classification: An Overview," Int. J. Data Warehousing Mining, vol. 3, no. 3, pp. 1–13, 2007, doi: 10.4018/jdwm.2007070101.